# 02. 투자 논거 vs 반대 논거 토론

**나는 BULL(매수 논거를 제시하는 애널리스트), AI는 BEAR(의심 많은 PM).** 토론이 끝나면 토론에 참여하지 않은 **독립 심판**이 양쪽의 근거 규율을 채점한다.

### 규칙 (코드가 강제함)
- 근거는 01 노트북이 만든 재무 지표(`GM_FY2026` 등)와, 있으면 `run.py`로 공시 원문에서 뽑은 팩트(`F1`, `F2` …)뿐이다.
- AI의 반박은 **근거 id를 반드시 인용**해야 한다. 인용이 없거나 존재하지 않는 id를 쓰면 코드가 그 반박을 **걸러내고 따로 표시**한다.
- 내가 `[id]`로 인용한 근거도 똑같이 검사한다.
- 밸류에이션·주가는 근거에 없다. AI가 지어내지 못하게 막아 두었다.
- **최종 판단은 사람이 한다.** 이 노트북의 결과물은 '토론 전 관점 → 토론 → 토론 후 관점'의 기록이다.

### 실행 모드
- `ANTHROPIC_API_KEY`가 있으면(환경변수 또는 저장소 루트의 `.env`) Claude API로 실제 토론한다.
- 없으면 **연습 모드**: 가짜 응답으로 흐름과 자동 검증만 확인한다.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from research import financials as fin, playbook

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"font.family": ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"],
                     "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"   # 범주 색 고정 순서
from research.debate import Debate, load_filing_facts

TICKER = "3778.T"
COMPANY = "SAKURA internet (さくらインターネット)"

ev_file = ROOT / "data" / "evidence" / f"{TICKER}.json"
if not ev_file.exists():
    raise FileNotFoundError("01_fundamentals.ipynb를 먼저 실행하세요.")
saved = json.loads(ev_file.read_text(encoding="utf-8"))
evidence = saved["evidence"] + load_filing_facts(TICKER.split(".")[0])

pb = playbook.load(saved.get("playbook", "generic"))
d = Debate(COMPANY, TICKER, evidence, language="Korean", bear_notes=playbook.bear_notes_text(pb))
print("모드:", "연습 모드 (API 호출 없음)" if d.dry_run else "Claude API", "| 근거 수:", len(d.evidence))

모드: 연습 모드 (API 호출 없음) | 근거 수: 64


## 근거 목록

토론에서 인용할 수 있는 id들이다. 최근 2개년만 보여 준다 (전체는 `d.evidence`).

In [2]:
recent = [k for k in d.evidence if k.endswith(("FY2025", "FY2026")) or k.startswith("F")]
pd.DataFrame({"근거": [d.evidence[k] for k in recent]}, index=recent)

,근거
REV_FY2025,매출 FY2025(03): 314.1억 JPY (전년 218.3억 JPY)
REV_FY2026,매출 FY2026(03): 353.0억 JPY (전년 314.1억 JPY)
REV_G_FY2025,매출 성장률 FY2025(03): 43.9% (전년 5.8%)
REV_G_FY2026,매출 성장률 FY2026(03): 12.4% (전년 43.9%)
GM_FY2025,매출총이익률 FY2025(03): 35.8% (전년 26.3%)
GM_FY2026,매출총이익률 FY2026(03): 22.5% (전년 35.8%)
OPM_FY2025,영업이익률 FY2025(03): 13.2% (전년 4.1%)
OPM_FY2026,영업이익률 FY2026(03): -1.1% (전년 13.2%)
EBITDA_M_FY2025,EBITDA 마진 FY2025(03): 28.7% (전년 19.0%)
EBITDA_M_FY2026,EBITDA 마진 FY2026(03): 24.1% (전년 28.7%)


## 01에서 찾은 질문들 (토론 재료)

In [3]:
icon = {"red": "🔴", "yellow": "🟡"}
display(Markdown("\n".join(f"- {icon[q['level']]} `{q['evidence_id']}` {q['question']}" for q in saved["questions"])))

- 🟡 `GM_FY2025` FY2025(03) 매출총이익률이 26.3%→35.8%로 +9.5%p 변했다. 가격(단가) 때문인가, 원가(전력·감가상각·외주) 때문인가, 매출 구성 변화 때문인가?
- 🟡 `OPM_FY2025` FY2025(03) 영업이익률 4.1%→13.2%. 매출총이익 변화와 판관비 중 어느 쪽이 컸나?
- 🟡 `SHARE_G_FY2025` FY2025(03) 발행주식수 +12.0%. 증자 자금은 어디에 썼고, 희석을 넘는 이익 증가가 있었나?
- 🔴 `PPE_TURN_FY2025` FY2025(03) 유형자산 회전율 1.31x→0.94x. 자산이 매출보다 훨씬 빨리 늘었다. 가동률은? 계약된 수요가 있는 투자인가?
- 🟡 `GM_FY2026` FY2026(03) 매출총이익률이 35.8%→22.5%로 -13.2%p 변했다. 가격(단가) 때문인가, 원가(전력·감가상각·외주) 때문인가, 매출 구성 변화 때문인가?
- 🟡 `OPM_FY2026` FY2026(03) 영업이익률 13.2%→-1.1%. 매출총이익 변화와 판관비 중 어느 쪽이 컸나?
- 🔴 `FCF_M_FY2026` FY2026(03) FCF 마진 -88.4%. 마이너스 FCF는 언제 끝나고, 그때까지 자금은 충분한가?
- 🟡 `DA_REV_FY2026` FY2026(03) 감가상각비/매출이 15.2%→21.9%로 올랐다. 새로 들인 설비(GPU)가 매출을 따라오기 전에 비용부터 반영되는 중인가? 내용연수 가정은 몇 년인가?
- 🔴 `PPE_TURN_FY2026` FY2026(03) 유형자산 회전율 0.94x→0.76x. 자산이 매출보다 훨씬 빨리 늘었다. 가동률은? 계약된 수요가 있는 투자인가?
- 🟡 `CAPEX_REV_FY2026` FY2026(03) 설비투자가 매출의 106.1%다. 투자 회수 기간과 자금 조달(차입·증자·보조금)은?

## 토론 전 내 관점

한두 문장. 토론이 끝난 뒤 이것과 비교한다.

In [4]:
VIEW_BEFORE = """(예시. 본인 관점으로 바꾸세요)
사쿠라인터넷은 정부 지원을 받는 일본 유일 규모의 국산 GPU 클라우드로, 지금의 이익 악화는 선행 투자에 따른 감가상각 때문이며 가동률이 올라오면 마진이 회복된다."""

## Round 1: 매수 논거 제시

**쓰는 법:** 주장마다 근거 id를 `[ ]`로 붙인다. 강한 논거는 '좋아 보인다'가 아니라 *시장이 무엇을 놓치고 있고, 무엇이 그 간극을 메울지*를 말한다.

In [5]:
pitch1 = """(예시. 본인 논거로 바꾸세요)
1) 매출은 2년 동안 크게 늘었다 [REV_G_FY2025][REV_G_FY2026]. GPU 수요가 실제로 매출로 잡히고 있다.
2) 영업이익 적자는 감가상각 부담 때문이고 [DA_REV_FY2026], 감가상각 전 이익인 EBITDA 마진은 아직 20%대다 [EBITDA_M_FY2026].
3) 설비투자가 매출보다 크지만 [CAPEX_REV_FY2026] 성장 투자 단계이므로 정상이다 [CAPEX_DA_FY2026]."""

r1 = d.bull(pitch1)
display(Markdown(d.round_md(r1, 1)))

### Round 1

**BULL (나)**

(예시. 본인 논거로 바꾸세요)
1) 매출은 2년 동안 크게 늘었다 [REV_G_FY2025][REV_G_FY2026]. GPU 수요가 실제로 매출로 잡히고 있다.
2) 영업이익 적자는 감가상각 부담 때문이고 [DA_REV_FY2026], 감가상각 전 이익인 EBITDA 마진은 아직 20%대다 [EBITDA_M_FY2026].
3) 설비투자가 매출보다 크지만 [CAPEX_REV_FY2026] 성장 투자 단계이므로 정상이다 [CAPEX_DA_FY2026].

**BEAR (AI PM)**

- **성장성**: (연습 모드) 투자가 매출보다 빨리 늘고 있다.  
  근거: PPE_TURN_FY2024, PPE_TURN_FY2025
    - `PPE_TURN_FY2024` 유형자산 회전율(매출/순유형자산) FY2024(03): 1.31x (전년 1.40x)
    - `PPE_TURN_FY2025` 유형자산 회전율(매출/순유형자산) FY2025(03): 0.94x (전년 1.31x)

*근거 id가 없거나 틀려서 자동으로 걸러진 반박 1개:*
- ~~(연습 모드) 이 반박은 없는 근거를 인용해서 자동으로 걸러져야 한다.~~ (인용: ['MADE_UP_2099'])

**인정한 부분:** (연습 모드) 매출 성장 자체는 수치로 확인된다.

**추가로 필요한 자료:** GPU 클러스터 가동률

**가장 어려운 질문:** (연습 모드) 투자금을 회수하는 시점은 언제이고, 그 전에 자금이 부족해지지 않는가?


## Round 2: 가장 어려운 질문에 답하기

위에서 BEAR가 던진 **가장 어려운 질문**과 반박에 답한다. 답할 수 없으면 '모른다, 이 자료가 필요하다'고 쓰는 것도 좋은 답이다. 실제 투자위원회에서 그게 더 신뢰를 얻는다.

In [6]:
pitch2 = """(예시. BEAR의 질문을 읽고 본인 답으로 바꾸세요)
FCF가 크게 마이너스인 건 맞다 [FCF_M_FY2026]. 다만 2025년 증자로 자기자본을 늘렸고 [SHARE_G_FY2025] 부채비율은 아직 1배 수준이다 [DEBT_EQUITY_FY2026].
가동률과 계약 기간은 재무제표로 확인할 수 없다. 유가증권보고서와 실적발표 Q&A에서 확인해야 한다."""

r2 = d.bull(pitch2)
display(Markdown(d.round_md(r2, 2)))

### Round 2

**BULL (나)**

(예시. BEAR의 질문을 읽고 본인 답으로 바꾸세요)
FCF가 크게 마이너스인 건 맞다 [FCF_M_FY2026]. 다만 2025년 증자로 자기자본을 늘렸고 [SHARE_G_FY2025] 부채비율은 아직 1배 수준이다 [DEBT_EQUITY_FY2026].
가동률과 계약 기간은 재무제표로 확인할 수 없다. 유가증권보고서와 실적발표 Q&A에서 확인해야 한다.

**BEAR (AI PM)**

- **성장성**: (연습 모드) 투자가 매출보다 빨리 늘고 있다.  
  근거: PPE_TURN_FY2024, PPE_TURN_FY2025
    - `PPE_TURN_FY2024` 유형자산 회전율(매출/순유형자산) FY2024(03): 1.31x (전년 1.40x)
    - `PPE_TURN_FY2025` 유형자산 회전율(매출/순유형자산) FY2025(03): 0.94x (전년 1.31x)

*근거 id가 없거나 틀려서 자동으로 걸러진 반박 1개:*
- ~~(연습 모드) 이 반박은 없는 근거를 인용해서 자동으로 걸러져야 한다.~~ (인용: ['MADE_UP_2099'])

**인정한 부분:** (연습 모드) 매출 성장 자체는 수치로 확인된다.

**추가로 필요한 자료:** GPU 클러스터 가동률

**가장 어려운 질문:** (연습 모드) 투자금을 회수하는 시점은 언제이고, 그 전에 자금이 부족해지지 않는가?


라운드를 더 하려면 아래 셀을 복사해서 쓰면 된다.

```python
r3 = d.bull("""...""")
display(Markdown(d.round_md(r3, 3)))
```

## 독립 심판 평가

토론에 참여하지 않은 별도의 호출이 양쪽 주장을 근거 기준으로 판정한다. 심판이 인용한 id도 코드가 다시 검사한다.

In [7]:
j = d.judge()
display(Markdown(d.judge_md()))

## 독립 심판 평가

근거 규율 점수: BULL 2/5, BEAR 4/5

### BULL 주장
| 주장 | 근거 | 판정 | 코멘트 |
|---|---|---|---|
| (연습 모드) 성장 논거 | PPE_TURN_FY2024 | overreach | 한 해 수치로 추세를 말함 |

### BEAR 주장
| 주장 | 근거 | 판정 | 코멘트 |
|---|---|---|---|
| (연습 모드) 현금흐름 악화 | PPE_TURN_FY2025 | supported | 수치가 직접 뒷받침 |
| (연습 모드) 존재하지 않는 근거 | FAKE_ID | unsupported | [자동 검증] 존재하지 않는 근거 id: ['FAKE_ID']. 검증 테스트용 |

**가장 강한 BULL 논거:** (연습 모드)

**가장 강한 BEAR 논거:** (연습 모드)

### 아직 풀리지 않은 질문
- GPU 가동률 → 확인할 곳: 실적발표 Q&A

### 다음 리서치 단계
1. 유가증권보고서의 설비투자 계획 확인
2. 고객 계약 기간 확인
3. 감가상각 내용연수 확인

## 토론 후 내 관점, 그리고 저장

**무엇이 바뀌었고 왜 바뀌었는지**를 쓴다. 바뀐 게 없다면 어떤 반박을 왜 받아들이지 않았는지를 쓴다. 이 기록이 이 프로젝트의 핵심 산출물이다.

In [8]:
VIEW_AFTER = """(예시. 본인 생각으로 바꾸세요)
감가상각 때문이라는 설명만으로는 부족하다. 유형자산 회전율이 계속 떨어지고 있어서 [PPE_TURN_FY2026] '가동률이 오르면 회복'이라는 가정을 숫자로 확인하기 전까지는 확신도를 낮춘다. 다음 단계: 유가증권보고서에서 GPU 설비 가동률·고객 계약 기간·보조금 회계 처리 확인."""

path = d.save(VIEW_BEFORE, VIEW_AFTER)
print("저장:", path.relative_to(ROOT))

저장: outputs\debates\3778.T_20261125_1450.md
